* [#29](https://github.com/salgo60/ifkdb/issues/29)
* denna notebook [29_linkroot_ifkdb.ipynb](https://github.com/salgo60/ifkdb/blob/main/Notebook/29_linkroot_ifkdb.ipynb)
  * rapport [ifkdb_linkroot_report_20260303.html](https://salgo60.github.io/ifkdb/Notebook/ifkdb_linkroot_report_20260303.html)
 
Kör en v2 rapport efter att vi har en lösning...
* rapport V2 [ifkdb_linkroot_report_20260303_v2.html](https://salgo60.github.io/ifkdb/Notebook/ifkdb_linkroot_report_20260303_v2.html)


In [1]:
from datetime import datetime
import platform
import sys
import os

# Start timer
start_time = datetime.now()
print(f"Start time: {start_time:%Y-%m-%d %H:%M:%S}")


Start time: 2026-03-03 12:44:04


In [2]:
import requests

def check_url(url):
    r = requests.get(url, timeout=10, allow_redirects=True)
    
    final_url = r.url
    status = r.status_code
    
    # Hard 404
    if status == 404:
        return "hard 404", final_url
    
    # Redirect to homepage = broken routing
    if final_url.rstrip("/") == "https://ifkdb.se":
        return "redirect to homepage (broken)", final_url
    
    # Soft 404 check (if site uses text message)
    if status == 200 and "hittas inte" in r.text.lower():
        return "soft 404", final_url
    
    # Looks OK only if still on /spelare/
    if "/spelare/" in final_url:
        return "ok", final_url
    
    return f"other({status})", final_url


base = "https://ifkdb.se/spelare/"
id = "SamuelWowoah_840"

url1 = base + id
print(check_url(url1))

url2 = url1.replace("SamuelWowoah","Samuel-Wowoah")
print(check_url(url2))

('ok', 'https://ifkdb.se/spelare/Samuel-Wowoah_840')
('ok', 'https://ifkdb.se/spelare/Samuel-Wowoah_840')


In [3]:
from tqdm import tqdm
import requests
import pandas as pd
import time
from datetime import datetime
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# -------------------------
# 1. SPARQL QUERY
# -------------------------

SPARQL_ENDPOINT = "https://query.wikidata.org/sparql"

query = """
SELECT ?item ?itemLabel ?ifkdbid ?ifkdb WHERE {
  ?item wdt:P11905 ?ifkdbid.
  BIND(URI(CONCAT("https://ifkdb.se/spelare/", ?ifkdbid)) AS ?ifkdb)
  SERVICE wikibase:label { bd:serviceParam wikibase:language "sv,en,mul". }
}
"""

headers = {
    "Accept": "application/sparql-results+json",
    "User-Agent": "IFKDB-Link-Checker/1.0 (research QA)"
}

response = requests.get(SPARQL_ENDPOINT, params={"query": query}, headers=headers)
sparql_data = response.json()

rows = []

for row in sparql_data["results"]["bindings"]:
    rows.append({
        "item": row["item"]["value"],
        "QID": row["item"]["value"].split("/")[-1],
        "label": row.get("itemLabel", {}).get("value", ""),
        "ifkdb": row["ifkdb"]["value"]
    })

wd_df = pd.DataFrame(rows)

# -------------------------
# 2. Friendly session setup
# -------------------------

session = requests.Session()

retry = Retry(
    total=3,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
)

adapter = HTTPAdapter(max_retries=retry)
session.mount("http://", adapter)
session.mount("https://", adapter)

session.headers.update({
    "User-Agent": "IFKDB-Link-Checker/1.0 (contact: salgo60@msn.com)"
})

In [ ]:

# -------------------------
# 3. URL CHECK FUNCTION
# -------------------------

def check_url(url):
    try:
        r = session.get(url, timeout=8, allow_redirects=True)
        final_url = r.url
        status = r.status_code

        if status == 404:
            return "hard 404", final_url
        
        if final_url.rstrip("/") == "https://ifkdb.se":
            return "redirect to homepage", final_url

        if status == 200 and "/spelare/" in final_url:
            return "ok", final_url

        return f"other({status})", final_url

    except Exception as e:
        return f"error", ""

# -------------------------
# 4. LOOP (with delay)
# -------------------------

results = []

for _, row in tqdm(wd_df .iterrows(), total=len(wd_df ), desc="Checking IFKDB URLs"):
    status, final_url = check_url(row["ifkdb"])
    
    if status != "ok":
        results.append({
            "Player": row["label"],
            "Original URL": row["ifkdb"],
            "Final URL": final_url,
            "Status": status
        })

    #time.sleep(0.5) 
error_df = pd.DataFrame(results)

# -------------------------
# 5. Make links clickable
# -------------------------

def make_clickable(url):
    if url:
        return f'<a href="{url}" target="_blank">{url}</a>'
    return ""

if not error_df.empty:
    error_df["Original URL"] = error_df["Original URL"].apply(make_clickable)
    error_df["Final URL"] = error_df["Final URL"].apply(make_clickable)



Checking IFKDB URLs:  71%|██████████████▏     | 412/582 [14:24<04:18,  1.52s/it]

In [ ]:
import re

error_df["ifkdb"] = error_df["Original URL"].str.extract(r'href="([^"]+)"')

In [ ]:
error_df = error_df.merge(
    wd_df[["QID", "ifkdb"]],
    on="ifkdb",
    how="left"
)

In [ ]:
error_df.info() 
error_df.head()

In [ ]:
# -------------------------
# 6. HTML REPORT
# -------------------------

today = datetime.now().strftime("%Y%m%d")
filename = f"ifkdb_linkroot_report_{today}_v2.html"

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")

# -------------------------
# Förbered DataFrame
# -------------------------

# Ta bort kolumn ifkdb om den finns
if "ifkdb" in error_df.columns:
    error_df = error_df.drop(columns=["ifkdb"])

# Gör QID klickbar
if "QID" in error_df.columns:
    error_df["QID"] = error_df["QID"].apply(
        lambda q: f'<a href="https://www.wikidata.org/wiki/{q}" target="_blank">{q}</a>'
        if pd.notna(q) else ""
    ) 
html = f"""
<html>
<head>
<meta charset="utf-8">
<title>IFKDB Länkrapport</title>
<style>
body {{ font-family: Arial; }}
table {{ border-collapse: collapse; width: 100%; }}
th, td {{ border: 1px solid #ccc; padding: 6px; }}
th {{ background-color: #f2f2f2; }}
tr:nth-child(even) {{ background-color: #fafafa; }}
a {{ text-decoration: none; color: blue; }}
</style>
</head>
<body>
<h1>IFKDB Länkrapport</h1>
<p>Genererad: {timestamp}</p>
<p>Antal fel: {len(error_df)}</p>
<p>Min issue: <a href="https://github.com/salgo60/ifkdb/issues/29" target_blank>#29</a></p>

{error_df.to_html(index=False, escape=False)}
</body>
</html>
"""

with open(filename, "w", encoding="utf-8") as f:
    f.write(html)

print(f"Rapport skapad: {filename}")

In [ ]:
# End timer
end_time = datetime.now()
duration = end_time - start_time

print("\n--- Execution Summary ---")
print(f"End time:      {end_time:%Y-%m-%d %H:%M:%S}")
print(f"Duration:      {duration}")
print(f"Total seconds: {duration.total_seconds():.2f}")
print(f"Python ver:    {sys.version.split()[0]}")
print(f"Platform:      {platform.system()} {platform.release()}")
print(f"Process ID:    {os.getpid()}")